# 09 — Optional English-trained anchor for forgetting

**Purpose:** Create one shared English-specialized model from which BOTH LoRA and full Urdu adaptation begin.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


A direct before/after English score from the original Qwen model is only a language-retention check, not necessarily meaningful *catastrophic forgetting*. For a stronger claim, establish English ability first using the SIB-200 English training split. Then save a **shared English-trained checkpoint**, and start both Urdu training methods from that checkpoint. The resulting comparison isolates what Urdu adaptation changes.

In [ ]:
import torch
RUN_ANCHOR=False  # English training consumes additional GPU time
if RUN_ANCHOR:
    assert torch.cuda.is_available()
    r=train_run('lora','sib200',size=10,seed=42,lang='en',rank=8,
                run_name='sib200_english_anchor_lora_r8')
    print('English adapter ready:',r['saved_path'])

## Merge the English LoRA adapter for a common starting point

This modifies the base weights once, creating a new English-adapted checkpoint. Then both Urdu methods begin from the same saved merged model. **Do not mix the English adapter and Urdu adapter in a way that gives only one method a hidden advantage.**

In [ ]:
MERGE_ANCHOR=False
if MERGE_ANCHOR:
    from peft import PeftModel
    r=read_json(ROOT/'outputs/checkpoints/sib200_english_anchor_lora_r8/complete.json')
    m,t=load_model(r['base_model'],adapter_path=r['saved_path'])
    merged=m.merge_and_unload()
    target=ROOT/'outputs/english_anchor_merged'
    target.mkdir(exist_ok=True)
    merged.save_pretrained(str(target),safe_serialization=True)
    t.save_pretrained(str(target))
    print('Saved common anchor:',target)
    print('Set MODEL_OVERRIDE=str(ROOT/"outputs/english_anchor_merged") in notebooks 03 and 04.')

### Cross-lingual evaluation

Use notebook 05 with `LANG='en'` and `SPLIT='test'` to compare the English anchor **before** Urdu adaptation and each Urdu-trained model **after** adaptation. Your final report should distinguish English retention from broader catastrophic forgetting.